# Hello LangGraph
이곳에서 LangGraph 에이전트 테스트를 진행할 수 있습니다.

In [13]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import  TypedDict
from typing import Annotated
import operator
from datetime import datetime
from typing import Literal
from typing import Union
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool


llm = init_chat_model("openai:gpt-4o-mini")


In [14]:
class State(MessagesState):
    custom_stuff : str


graph_builder = StateGraph(State)


In [15]:
@tool
def get_weather(city : str) -> str:
    """Get the current weather for a given city."""
    return f"The current weather in {city} is sunny with a temperature of 25°C."

llm_with_tools = llm.bind_tools([get_weather])

def chatbot(state : State):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages" : [response]
    }

tool_node = ToolNode(
    tools=[
        get_weather
    ]
)

In [16]:


graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

graph.invoke(
    {
        "messages" : [
            {"role":"user", "content":"What's the weather in Seoul?"}
        ]
    }
)


{'messages': [HumanMessage(content="What's the weather in Seoul?", additional_kwargs={}, response_metadata={}, id='b43b6003-2932-4cd3-b18b-52b106591ea1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 52, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DP86Lt5Auaefa1ZFM3jo29PuxqwKz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3f31-a5f9-7342-a45a-7e5d2f4f1986-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Seoul'}, 'id': 'call_S98PzcbjmruQL0U52zGWoJNk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_